In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision
import os
import matplotlib.pyplot as plt

In [ ]:
sample_dir = 'samples'
if not os.path.exists(sample_dir):
  os.makedirs(sample_dir)

In [ ]:
# z = N(0,1) --> G(z) --> [-1,-1] --> D(G(z))

# Output of Discrimator : D(image) outputs value in [0,1] which is probability of image being real image.
# Task of Discrimator : D(x) --> close to 1, && 1-D(G(z)) --> close 1

# in GAN we alternate between D and G
# what if I train my D completely first : D(G(z)) --> 0

latent_size = 64
hidden_size = 256
image_size = 784 # 28X28 (for mnist)
num_epochs = 100

batch_size = 100


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:

def load_mnist_data(batch_size):

  transform = transforms.Compose([
    transforms.ToTensor(), # Image [0-255] -> [0,1]
    transforms.Normalize(mean=[0.5],std=[0.5])] # Normalize [0,1] with (mean=0.5,std=0.5) -> [-1,1]
  )
  mnist = torchvision.datasets.FashionMNIST(
      root = "./data/",
      train = True,
      transform = transform,
      download=True
  )

  data_loader = torch.utils.data.DataLoader(
      dataset = mnist,
      batch_size = batch_size,
      shuffle=True
  )

  return data_loader

In [ ]:
data_loader = load_mnist_data(batch_size)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 210kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.93MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.3MB/s]


In [ ]:
class Generator(nn.Module):
  def __init__(self,latent_size,hiddle_size,image_size):
    super(Generator,self).__init__()

    self.model = nn.Sequential(
        nn.Linear(latent_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,image_size),
        nn.Tanh()
    )

  def forward(self,z):
    return self.model(z)

generator = Generator(latent_size,hidden_size,image_size).to(device)

In [ ]:
print(generator)

Generator(
  (model): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=256, out_features=784, bias=True)
    (5): Tanh()
  )
)


In [ ]:
print(f'Total Parameters: {sum(p.numel() for p in generator.parameters())}')

Total Parameters: 283920


In [ ]:
class Discriminator(nn.Module):
  def __init__(self,image_size,hidden_size):
    super(Discriminator,self).__init__()

    self.model = nn.Sequential(
        nn.Linear(image_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,1),
        nn.Sigmoid()
    )

  def forward(self,x):
    return self.model(x)

discriminator = Discriminator(image_size,hidden_size).to(device)

In [ ]:
print(discriminator)

Discriminator(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=256, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [ ]:
print(f'Total Parameters: {sum(p.numel() for p in discriminator.parameters())}')

Total Parameters: 267009


In [ ]:
criterion = nn.BCELoss()

d_optimizer_fixed = torch.optim.Adam(discriminator.parameters(),lr=0.0002, betas=(0.5,0.999))
g_optimizer_fixed = torch.optim.Adam(generator.parameters(),lr=0.0002, betas=(0.5,0.999))



In [ ]:
def denorm(x):
  # x_norm = (x-0.5)/0.5, x = (x_norm * 0.5) + 0.5
  out = (x + 1)/2
  return out.clamp(0,1)

In [ ]:
def reset_grad():
  d_optimizer_fixed.zero_grad()
  g_optimizer_fixed.zero_grad()

In [ ]:
total_step = len(data_loader)
d_losses = []
g_losses = []
num_epochs = 100

for epoch in range(num_epochs):
  epoch_d_loss = 0
  epoch_g_loss = 0

  for i, (images,label) in enumerate(data_loader):
    images = images.reshape(batch_size,-1).to(device)

    real_labels = torch.ones(batch_size,1).to(device)
    fake_labels = torch.zeros(batch_size,1).to(device)

    outputs = discriminator(images) # [0-1] passing the real image to learn
    d_loss_real = criterion(outputs,real_labels)
    real_score = outputs

    z = torch.randn(batch_size,latent_size).to(device)
    fake_images = generator(z)

    outputs = discriminator(fake_images) # outputs should be close to 0
    d_loss_fake = criterion(outputs,fake_labels)
    fake_score = outputs

    d_loss = d_loss_real + d_loss_fake

    reset_grad()
    d_loss.backward()
    d_optimizer_fixed.step() # update our parameters


# --------------------Generator -------------------------------------------
    # Generator wants to fool discriminator
    z = torch.randn(batch_size,latent_size).to(device)
    fake_images = generator(z)

    outputs = discriminator(fake_images)
    g_loss = criterion(outputs,real_labels) # want to change to parameters of generator

    reset_grad()
    g_loss.backward()
    g_optimizer_fixed.step()


    epoch_d_loss += d_loss.item()
    epoch_g_loss += g_loss.item()

    if (i + 1) % 200 == 0:
      print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{total_step}], '
            f'D_loss: {d_loss.item():.4f}, G_loss: {g_loss.item():.4f}, '
            f'D(x): {real_score.mean().item():.2f}, D(G(z)): {fake_score.mean().item():.2f}')


    d_losses.append(epoch_d_loss / total_step)
    g_losses.append(epoch_g_loss / total_step)

    fake_images_grid = fake_images.reshape(fake_images.size(0), 1, 28, 28)
    torchvision.utils.save_image(
        denorm(fake_images_grid),
        os.path.join(sample_dir, f'fake_images-{epoch+1}.png')
        )






Epoch [1/100], Step [200/600], D_loss: 0.4443, G_loss: 1.6562, D(x): 0.95, D(G(z)): 0.31
Epoch [1/100], Step [400/600], D_loss: 0.7233, G_loss: 1.9320, D(x): 0.75, D(G(z)): 0.24
Epoch [1/100], Step [600/600], D_loss: 0.7525, G_loss: 3.1160, D(x): 0.82, D(G(z)): 0.40
Epoch [2/100], Step [200/600], D_loss: 0.3898, G_loss: 2.9166, D(x): 0.86, D(G(z)): 0.16
Epoch [2/100], Step [400/600], D_loss: 0.6340, G_loss: 1.4674, D(x): 0.70, D(G(z)): 0.16
Epoch [2/100], Step [600/600], D_loss: 0.6449, G_loss: 2.0599, D(x): 0.76, D(G(z)): 0.23
Epoch [3/100], Step [200/600], D_loss: 0.5525, G_loss: 2.2492, D(x): 0.76, D(G(z)): 0.11
Epoch [3/100], Step [400/600], D_loss: 0.5348, G_loss: 3.6482, D(x): 0.86, D(G(z)): 0.25
Epoch [3/100], Step [600/600], D_loss: 0.5016, G_loss: 2.3998, D(x): 0.81, D(G(z)): 0.16
Epoch [4/100], Step [200/600], D_loss: 0.6889, G_loss: 2.1488, D(x): 0.70, D(G(z)): 0.06
Epoch [4/100], Step [400/600], D_loss: 0.7919, G_loss: 2.2439, D(x): 0.79, D(G(z)): 0.32
Epoch [4/100], Step [